# Figure 8

In [1]:
from pathlib import Path
import csv
import json

import matplotlib.pyplot as plt
import numpy as np
from scipy.interpolate import griddata

plt.rcParams.update({
    'font.family': 'sans-serif',
    'font.sans-serif': ['Arial', 'Helvetica', 'DejaVu Sans'],
    'font.size': 18,
    'font.weight': 'bold',
    'axes.labelweight': 'bold',
    'axes.titleweight': 'bold',
    'axes.unicode_minus': False,
    'axes.linewidth': 0.8,
    'xtick.direction': 'out',
    'ytick.direction': 'out',
})

ROOT = Path.cwd()
FIGURE_DIR = ROOT / 'Figure'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

CASE_SPECS = {
    'Hannah': {'inversion_dir': 'Hannah_Inversion_GPT'},
    'Iowa': {'inversion_dir': 'Iowa_Inversion_GPT'},
}
COMBINED_OUTPUT = FIGURE_DIR / 'Figure8_datafit_assessment.png'
PROPERTIES = {
    'gravity': {'label': 'Gravity', 'prediction_file': 'dpred_gravity.npy'},
    'magnetics': {'label': 'Magnetics', 'prediction_file': 'dpred_magnetics.npy'},
}

def nice_limit(value):
    """Round a colour-bar range to a compact readable limit."""
    if value <= 0:
        return 1.0
    scale = 10.0 ** np.floor(np.log10(value))
    for candidate in (1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0, 6.0, 8.0, 10.0):
        if value / scale <= candidate:
            return candidate * scale
    return 10.0 * scale

def load_property(case, property_name):
    """Load coordinates, observations, predictions, residuals, and units."""
    inversion_dir = ROOT / CASE_SPECS[case]['inversion_dir']
    observed_data = np.loadtxt(inversion_dir / 'observed_data' / f'{property_name}.obs', ndmin=2)
    predicted = np.load(inversion_dir / 'inversion_result' / PROPERTIES[property_name]['prediction_file']).reshape(-1)
    if observed_data.shape[1] < 4 or predicted.size != observed_data.shape[0]:
        raise ValueError(f'{case} {property_name}: observation and prediction arrays are not aligned.')
    with (inversion_dir / 'inversion_params.json').open(encoding='utf-8') as handle:
        gravity_component = json.load(handle).get('gravity_component', '').lower()
    unit = 'nT' if property_name == 'magnetics' else ('E' if gravity_component == 'gzz' else 'mGal')
    observed = observed_data[:, 3]
    residual = predicted - observed
    if not (np.isfinite(observed).all() and np.isfinite(predicted).all()):
        raise ValueError(f'{case} {property_name}: non-finite values found.')
    return observed_data[:, :2], observed, predicted, residual, unit

def interpolate_map(xy, values, grid_x, grid_y):
    mapped = griddata(xy, values, (grid_x, grid_y), method='linear')
    if np.isnan(mapped).any():
        mapped = np.where(np.isnan(mapped), griddata(xy, values, (grid_x, grid_y), method='nearest'), mapped)
    return mapped

def render_combined_comparison():
    columns = (
        ('Hannah', 'gravity', 'Hannah ISO'),
        ('Hannah', 'magnetics', 'Hannah TMI'),
        ('Iowa', 'gravity', 'Iowa Gzz'),
        ('Iowa', 'magnetics', 'Iowa TMI'),
    )
    rows = ('Observed', 'Predicted', 'Residual')
    field_limits = {
        ('Hannah', 'gravity'): 32.0, ('Hannah', 'magnetics'): 420.0,
        ('Iowa', 'gravity'): 80.0, ('Iowa', 'magnetics'): 1500.0,
    }
    field_ticks = {
        ('Hannah', 'gravity'): (-30.0, -15.0, 0.0, 15.0, 30.0),
        ('Hannah', 'magnetics'): (-400.0, -200.0, 0.0, 200.0, 400.0),
    }
    fig = plt.figure(figsize=(26.0, 19.0))
    grid = fig.add_gridspec(
        3, 4, left=0.10, right=0.99, bottom=0.075, top=0.92, wspace=0.28, hspace=0.10,
    )
    row_centres = (0.79, 0.50, 0.21)
    for row_title, row_centre in zip(rows, row_centres):
        fig.text(0.025, row_centre, row_title, ha='center', va='center', rotation=0,
                 fontsize=24, fontweight='bold')

    panel_axes = [[None for _ in columns] for _ in rows]
    for column_index, (case, property_name, column_title) in enumerate(columns):
        xy, observed, predicted, residual, unit = load_property(case, property_name)
        min_e, max_e = np.min(xy[:, 0]), np.max(xy[:, 0])
        min_n, max_n = np.min(xy[:, 1]), np.max(xy[:, 1])
        grid_e, grid_n = np.linspace(min_e, max_e, 260), np.linspace(min_n, max_n, 260)
        grid_x, grid_y = np.meshgrid(grid_e, grid_n)
        field_limit = field_limits[(case, property_name)]
        residual_limit = nice_limit(np.max(np.abs(residual)))
        panels = (
            (observed, 'coolwarm', -field_limit, field_limit),
            (predicted, 'coolwarm', -field_limit, field_limit),
            (residual, 'RdBu_r', -residual_limit, residual_limit),
        )

        for row_index, (row_title, (values, cmap, vmin, vmax)) in enumerate(zip(rows, panels)):
            panel_grid = grid[row_index, column_index].subgridspec(1, 2, width_ratios=(1.0, 0.06), wspace=0.05)
            ax = fig.add_subplot(panel_grid[0, 0])
            panel_axes[row_index][column_index] = ax
            mapped = interpolate_map(xy, values, grid_x, grid_y)
            image = ax.imshow(
                mapped, extent=[min_e / 1000, max_e / 1000, min_n / 1000, max_n / 1000],
                origin='lower', cmap=cmap, vmin=vmin, vmax=vmax, interpolation='bilinear',
            )
            ax.contour(grid_x / 1000, grid_y / 1000, mapped, levels=8, colors='black',
                       linewidths=0.45, linestyles='dashed', alpha=0.55)
            ax.set_aspect('auto')
            ax.tick_params(labelsize=20, width=1.0, length=4.5)
            if row_index == 2:
                ax.set_xlabel('Easting (km)', fontsize=23, fontweight='bold')
            else:
                ax.tick_params(labelbottom=False)
            if column_index == 0:
                ax.set_ylabel('Northing (km)', fontsize=23, fontweight='bold')
            else:
                ax.tick_params(labelleft=False)
            for tick_label in (*ax.get_xticklabels(), *ax.get_yticklabels()):
                tick_label.set_fontweight('bold')
            colorbar_axis = fig.add_subplot(panel_grid[0, 1])
            colorbar = fig.colorbar(image, cax=colorbar_axis)
            colorbar.set_ticks(field_ticks[(case, property_name)] if row_index < 2 and (case, property_name) in field_ticks else np.linspace(vmin, vmax, 5))
            colorbar.set_label(unit, fontsize=21, fontweight='bold')
            colorbar.ax.tick_params(labelsize=19, width=0.9, length=3.5)
            for tick_label in colorbar.ax.get_yticklabels():
                tick_label.set_fontweight('bold')

    fig.canvas.draw()
    for row_index, axes_in_row in enumerate(panel_axes):
        label_y = max(axis.get_position().y1 for axis in axes_in_row) + 0.006
        for column_index, axis in enumerate(axes_in_row):
            panel_label = chr(ord('a') + row_index * len(columns) + column_index)
            fig.text(axis.get_position().x0, label_y, f'({panel_label})', ha='left', va='bottom',
                     fontsize=23, fontweight='bold')
    header_y = max(axis.get_position().y1 for axis in panel_axes[0]) + 0.038
    for column_index, (_, _, column_title) in enumerate(columns):
        axis = panel_axes[0][column_index]
        bounds = axis.get_position()
        fig.text((bounds.x0 + bounds.x1) / 2.0, header_y, column_title, ha='center', va='bottom',
                 fontsize=24, fontweight='bold')

    fig.savefig(COMBINED_OUTPUT, dpi=600, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    return COMBINED_OUTPUT

def fit_metrics(case, property_name):
    _, observed, predicted, residual, unit = load_property(case, property_name)
    rmse = np.sqrt(np.mean(residual ** 2))
    ss_total = np.sum((observed - np.mean(observed)) ** 2)
    return {
        'case': case,
        'property': PROPERTIES[property_name]['label'],
        'unit': unit,
        'n': observed.size,
        'bias_predicted_minus_observed': np.mean(residual),
        'mae': np.mean(np.abs(residual)),
        'rmse': rmse,
        'nrmse_percent_of_observed_range': 100 * rmse / np.ptp(observed),
        'r_squared': 1 - np.sum(residual ** 2) / ss_total,
        'pearson_r': np.corrcoef(observed, predicted)[0, 1],
    }

combined_figure = render_combined_comparison()
metrics = [fit_metrics(case, property_name) for case in CASE_SPECS for property_name in PROPERTIES]
metrics_csv = FIGURE_DIR / 'Figure8_datafit_assessment_metrics.csv'
with metrics_csv.open('w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=list(metrics[0]))
    writer.writeheader()
    writer.writerows(metrics)

print(f'Wrote {combined_figure.relative_to(ROOT)}')
print(f'Wrote {metrics_csv.relative_to(ROOT)}')
for metric in metrics:
    print(metric)

Wrote Figure\FigureS1_Hannah_GM.png
Wrote Figure\FigureS2_Iowa_GM.png
Wrote Figure\Figure8_datafit_assessment_metrics.csv
{'case': 'Hannah', 'property': 'Gravity', 'unit': 'mGal', 'n': 669, 'bias_predicted_minus_observed': np.float64(-0.013000423113505058), 'mae': np.float64(0.20747682178679772), 'rmse': np.float64(0.24962684144209704), 'nrmse_percent_of_observed_range': np.float64(0.6555326718542464), 'r_squared': np.float64(0.9985014485224243), 'pearson_r': np.float64(0.9992746812639243)}
{'case': 'Hannah', 'property': 'Magnetics', 'unit': 'nT', 'n': 1875, 'bias_predicted_minus_observed': np.float64(0.23784878147957839), 'mae': np.float64(6.863448925481857), 'rmse': np.float64(9.972797210824867), 'nrmse_percent_of_observed_range': np.float64(1.0045458017758921), 'r_squared': np.float64(0.9955146274911076), 'pearson_r': np.float64(0.9978611246664691)}
{'case': 'Iowa', 'property': 'Gravity', 'unit': 'E', 'n': 1037, 'bias_predicted_minus_observed': np.float64(-0.0018091959804440929), 'm